# 🧠 Brain Tumor MRI — Data Exploration & Augmentation
**Sprint 1 | Yassine | PFA 3IIR**

**Dataset:** Brain Tumor MRI Dataset — Masoud Nickparvar (Kaggle)  
**Classes:** Glioma · Meningioma · Pituitary · No Tumor

## 1. Setup & Kaggle Authentication

In [ ]:
!pip install -q kaggle

In [ ]:
import os, json

# Paste your Kaggle API token here (do NOT commit this to GitHub)
KAGGLE_USERNAME = 'your_kaggle_username'  # replace with your Kaggle username
KAGGLE_KEY      = 'your_kaggle_api_key'   # replace with your Kaggle API key

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured.')

## 2. Download & Extract Dataset

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/dataset --unzip
print('Dataset downloaded and extracted.')

In [ ]:
import pathlib

# Detect dataset root (Training folder)
dataset_root = pathlib.Path('/content/dataset')

# Find Training directory
train_dir = None
for p in dataset_root.rglob('Training'):
    if p.is_dir():
        train_dir = p
        break

test_dir = None
for p in dataset_root.rglob('Testing'):
    if p.is_dir():
        test_dir = p
        break

print(f'Train dir: {train_dir}')
print(f'Test dir:  {test_dir}')
print('\nClasses found:', [d.name for d in train_dir.iterdir() if d.is_dir()])

## 3. EDA — Class Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_LABELS = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

def count_images(directory):
    counts = {}
    for cls in CLASSES:
        cls_path = directory / cls
        counts[cls] = len(list(cls_path.glob('*.jpg'))) + len(list(cls_path.glob('*.png')))
    return counts

train_counts = count_images(train_dir)
test_counts  = count_images(test_dir)

df = pd.DataFrame({
    'Class': CLASS_LABELS,
    'Train': list(train_counts.values()),
    'Test':  list(test_counts.values())
})

print(df.to_string(index=False))
print(f"\nTotal Train: {df['Train'].sum()} | Total Test: {df['Test'].sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for ax, split, col in zip(axes, ['Train', 'Test'], ['Train', 'Test']):
    bars = ax.bar(CLASS_LABELS, df[col], color=colors, edgecolor='black', linewidth=0.7)
    ax.set_title(f'{col} Set — Class Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Tumor Class')
    ax.set_ylabel('Number of Images')
    ax.set_ylim(0, df[col].max() * 1.2)
    for bar, val in zip(bars, df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. EDA — Sample MRI Images per Class

In [ ]:
import random
from PIL import Image
import numpy as np

SAMPLES_PER_CLASS = 5

fig, axes = plt.subplots(len(CLASSES), SAMPLES_PER_CLASS, figsize=(15, 12))
fig.suptitle('Sample MRI Images per Class', fontsize=15, fontweight='bold', y=1.01)

for row, (cls, label) in enumerate(zip(CLASSES, CLASS_LABELS)):
    images = list((train_dir / cls).glob('*.jpg')) + list((train_dir / cls).glob('*.png'))
    samples = random.sample(images, min(SAMPLES_PER_CLASS, len(images)))
    for col, img_path in enumerate(samples):
        img = Image.open(img_path).convert('RGB').resize((224, 224))
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(label, fontsize=11, fontweight='bold', rotation=90, labelpad=10)
            axes[row][col].yaxis.set_label_position('left')
            axes[row][col].yaxis.label.set_visible(True)

plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. EDA — Image Size & Pixel Intensity Stats

In [ ]:
# Check image sizes and pixel value distribution for one class
sample_imgs = list((train_dir / 'glioma').glob('*.jpg'))[:200]
pixel_means = []

for p in sample_imgs:
    arr = np.array(Image.open(p).convert('RGB').resize((224, 224))) / 255.0
    pixel_means.append(arr.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pixel_means, bins=30, color='steelblue', edgecolor='black')
ax.set_title('Pixel Mean Intensity Distribution (Glioma — 200 samples)', fontweight='bold')
ax.set_xlabel('Mean Pixel Value (normalized)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f'Mean: {np.mean(pixel_means):.4f} | Std: {np.std(pixel_means):.4f}')

## 6. Data Augmentation with ImageDataGenerator

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT  = 0.2
SEED       = 42

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# No augmentation on validation/test — only rescale
val_test_datagen = ImageDataGenerator(rescale=1./255)

print('ImageDataGenerators configured.')

In [ ]:
train_generator = train_datagen.flow_from_directory(
    str(train_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    str(train_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    str(test_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('\nClass indices:', train_generator.class_indices)
print(f'Train batches: {len(train_generator)} | Val batches: {len(val_generator)} | Test batches: {len(test_generator)}')

## 7. Visualize Augmented Images

In [ ]:
# Show original vs augmented for one sample
sample_path = list((train_dir / 'glioma').glob('*.jpg'))[0]
original = np.array(Image.open(sample_path).convert('RGB').resize((224, 224)))

aug_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

img_array = original.reshape((1,) + original.shape)
aug_iter  = aug_gen.flow(img_array, batch_size=1)

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(original)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

for i in range(1, 6):
    aug_img = next(aug_iter)[0].astype('uint8')
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Aug {i}')
    axes[i].axis('off')

plt.suptitle('Original vs Augmented (Glioma)', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Generators Config for Abdessattar (CNN Training)

In [ ]:
import json

config = {
    'img_size': list(IMG_SIZE),
    'batch_size': BATCH_SIZE,
    'val_split': VAL_SPLIT,
    'seed': SEED,
    'num_classes': 4,
    'class_indices': train_generator.class_indices,
    'train_samples': train_generator.samples,
    'val_samples': val_generator.samples,
    'test_samples': test_generator.samples,
    'augmentation': {
        'rotation_range': 20,
        'width_shift_range': 0.1,
        'height_shift_range': 0.1,
        'zoom_range': 0.2,
        'horizontal_flip': True,
        'brightness_range': [0.8, 1.2],
        'fill_mode': 'nearest'
    }
}

with open('/content/generators_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Config saved to /content/generators_config.json')
print(json.dumps(config, indent=2))

In [ ]:
# Download all outputs to your local machine
from google.colab import files

for f in ['class_distribution.png', 'sample_images.png', 'augmentation_preview.png', 'generators_config.json']:
    files.download(f'/content/{f}')

## ✅ Sprint 1 Summary

| Step | Status |
|------|--------|
| Dataset loaded from Kaggle | ✅ |
| Class distribution visualized | ✅ |
| Sample MRI images per class | ✅ |
| Pixel intensity stats | ✅ |
| ImageDataGenerator with augmentation | ✅ |
| Train / Val / Test generators ready | ✅ |
| Augmentation preview | ✅ |
| Config saved for CNN training | ✅ |

**Next:** Push to `feature/data-exploration` branch → Abdessattar uses generators for ResNet50 training.